In [1]:
!pip install pvlib

In [2]:
width_m, depth_m, height_m = 25, 25, 81
window_wall_ratio = 0.90
T_indoor_C = 24
T_outdoor_C = 35
U_wall = 0.25
U_roof = 0.20
U_glass = 1.4
ACH = 0.8
air_density = 1.2
air_cp = 1005
n_people = 300
person_heat_W = 200
light_total = 7
n_lights = 6500
electricity_price_eur_kwh = 0.34
cooling_cop = 2.5

In [3]:
footprint_m2 = width_m * depth_m
dT = T_outdoor_C - T_indoor_C

facade_area_by_orientation = {
    "North": width_m * height_m, "South": width_m * height_m,
    "East": depth_m * height_m, "West": depth_m * height_m,
}
glazing_area_by_orientation = {k: v * window_wall_ratio for k, v in facade_area_by_orientation.items()}
glazing_area = sum(glazing_area_by_orientation.values())
wall_area = 2 * (width_m + depth_m) * height_m * (1 - window_wall_ratio)
roof_area = footprint_m2
volume_m3 = footprint_m2 * height_m

Q_people = n_people * person_heat_W
Q_light = light_total * n_lights
Q_wall = U_wall * wall_area * dT
Q_roof = U_roof * roof_area * dT
Q_glass_cond = U_glass * glazing_area * dT
Q_vent = ACH * volume_m3 * air_density * air_cp * dT / 3600
Q_constant = Q_people + Q_light + Q_wall + Q_roof + Q_glass_cond + Q_vent

print(f"Glazing area: {glazing_area:.0f} m2")
print(f"Q_constant (non-solar): {Q_constant:.0f} W")

Glazing area: 7290 m2
Q_constant (non-solar): 370611 W


In [4]:
import numpy as np

tau_e_0 = {0: 0.450, 25: 0.416, 50: 0.364, 100: 0.213}
p = 2

def karlsson_roos_tau(theta_deg, tau0, q, p=2):
    if theta_deg >= 90:
        return 0.0
    z = theta_deg / 90
    a = 8
    b = 0.25 / q
    c = 1 - a - b
    alpha = 5.2 + 0.7*q
    beta = 2
    gamma = (5.26 + 0.06*p) + (0.73 + 0.14*q)
    factor = 1 - a*(z**alpha) - b*(z**beta) - c*(z**gamma)
    return tau0 * factor

# pazim tukaj vem
tau_60_deg_measured = {0: 0.430, 25: 0.395, 50: 0.350, 100: 0.240}

q_guess = {0: 2.19, 25: 2.15, 50: 2.24, 100: 3.92}   # <-- rocno spreminjaj, razlika prot nč more it

print(f"{'Stanje':>8} | {'tau(60) izmerjen':>18} | {'q':>6} | {'tau(60) napoved':>16} | {'razlika':>10}")
for state in [0, 25, 50, 100]:
    predicted = karlsson_roos_tau(60, tau_e_0[state], q_guess[state], p=p)
    diff = predicted - tau_60_deg_measured[state]
    print(f"{state:>7}% | {tau_60_deg_measured[state]:>18.3f} | {q_guess[state]:>6.2f} | {predicted:>16.3f} | {diff:>+10.3f}")

  Stanje |   tau(60) izmerjen |      q |  tau(60) napoved |    razlika
      0% |              0.430 |   2.19 |            0.430 |     -0.000
     25% |              0.395 |   2.15 |            0.395 |     -0.000
     50% |              0.350 |   2.24 |            0.350 |     +0.000
    100% |              0.240 |   3.92 |            0.240 |     +0.000


In [5]:
from scipy.integrate import quad

def tau_e_diffuse(state):
    tau0 = tau_e_0[state]
    q = q_guess[state]
    def integrand(phi_rad):
        phi_deg = np.degrees(phi_rad)
        return karlsson_roos_tau(phi_deg, tau0, q, p) * np.cos(phi_rad) * np.sin(phi_rad)
    integral, _ = quad(integrand, 0, np.pi/2)
    return 2 * integral

tau_e_dif = {state: tau_e_diffuse(state) for state in [0, 25, 50, 100]}

print("tau_e,dif (difuzna prepustnost):")
for state, val in tau_e_dif.items():
    print(f"  {state}%: {val:.4f}")

tau_e,dif (difuzna prepustnost):
  0%: 0.4243
  25%: 0.3909
  50%: 0.3446
  100%: 0.2231


In [6]:
rho_e_light = 0.275   # OCENA -- cakam na podatek proizvajalca
rho_e_dark = 0.08     # OCENA

rho_e = {state: rho_e_light + (state/100)*(rho_e_dark - rho_e_light) for state in [0, 25, 50, 100]}

print("rho_e (odbojnost, OCENA):")
for state, val in rho_e.items():
    print(f"  {state}%: {val:.3f}")

rho_e (odbojnost, OCENA):
  0%: 0.275
  25%: 0.226
  50%: 0.178
  100%: 0.080


In [7]:
Fi = 0.225   # OCENA, delez absorbirane energije, ki gre navznoter

def alpha_e_direct(theta_deg, state):
    tau = karlsson_roos_tau(theta_deg, tau_e_0[state], q_guess[state], p=p)
    return 1 - tau - rho_e[state]

def qi_direct(theta_deg, state):
    return Fi * alpha_e_direct(theta_deg, state)

def g_direct(theta_deg, state):
    tau = karlsson_roos_tau(theta_deg, tau_e_0[state], q_guess[state], p=p)
    qi = qi_direct(theta_deg, state)
    return tau + qi

print("DIREKTNI del -- g(theta) za stanje 100%:")
for theta in [0, 30, 60, 90]:
    print(f"  theta={theta}: g={g_direct(theta, 100):.4f}")

DIREKTNI del -- g(theta) za stanje 100%:
  theta=0: g=0.3721
  theta=30: g=0.3715
  theta=60: g=0.3931
  theta=90: g=0.2070


In [8]:
def alpha_e_diffuse(state):
    return 1 - tau_e_dif[state] - rho_e[state]

def qi_diffuse(state):
    return Fi * alpha_e_diffuse(state)

def g_diffuse(state):
    return tau_e_dif[state] + qi_diffuse(state)

g_dif = {state: g_diffuse(state) for state in [0, 25, 50, 100]}

print("DIFUZNI del -- g_dif za vsako stanje:")
for state, val in g_dif.items():
    print(f"  {state}%: {val:.4f}")

DIFUZNI del -- g_dif za vsako stanje:
  0%: 0.4919
  25%: 0.4770
  50%: 0.4521
  100%: 0.3799


In [9]:
import pvlib

FILEPATH_EPW = "SVN_LJ_Ljubljana-Bezigrad.140150_TMYx.2007-2021.epw"

epw_data, epw_meta = pvlib.iotools.read_epw(FILEPATH_EPW)
LAT = epw_meta['latitude']
LON = epw_meta['longitude']

print(f"Lokacija: {epw_meta['city']}, lat={LAT}, lon={LON}")
print(f"Stevilo urnih vrstic: {len(epw_data)}")

solar_position = pvlib.solarposition.get_solarposition(epw_data.index, LAT, LON)
print("Polozaj sonca izracunan za vseh 8760 ur.")

Lokacija: Ljubljana-Bezigrad, lat=46.0656, lon=14.5122
Stevilo urnih vrstic: 8760
Polozaj sonca izracunan za vseh 8760 ur.


In [10]:
facade_azimuths = {"North": 0, "East": 90, "South": 180, "West": 270}

# Manjkajoc podatek za 'perez' model -- izvenzemeljsko sevanje za vsako uro
dni_extra = pvlib.irradiance.get_extra_radiation(epw_data.index)

facade_irradiance = {}
for facade, az in facade_azimuths.items():
    total_irrad = pvlib.irradiance.get_total_irradiance(
        surface_tilt=90, surface_azimuth=az,
        solar_zenith=solar_position['apparent_zenith'],
        solar_azimuth=solar_position['azimuth'],
        dni=epw_data['dni'], ghi=epw_data['ghi'], dhi=epw_data['dhi'],
        dni_extra=dni_extra,
        model='perez'
    )
    aoi = pvlib.irradiance.aoi(
        surface_tilt=90, surface_azimuth=az,
        solar_zenith=solar_position['apparent_zenith'],
        solar_azimuth=solar_position['azimuth']
    )
    facade_irradiance[facade] = {
        'aoi': aoi,
        'poa_direct': total_irrad['poa_direct'].clip(lower=0),
        'poa_diffuse_total': (total_irrad['poa_diffuse'] + total_irrad.get('poa_ground_diffuse', 0)).clip(lower=0),
    }

print("Obsevanost vseh 4 fasad izracunana (direktni + difuzni del loceno).")

Obsevanost vseh 4 fasad izracunana (direktni + difuzni del loceno).


In [11]:
q_sol_results = {}

for state in [0, 25, 50, 100]:
    q_sol_by_facade = {}
    for facade in facade_azimuths:
        aoi = facade_irradiance[facade]['aoi']
        poa_direct = facade_irradiance[facade]['poa_direct']
        poa_diffuse = facade_irradiance[facade]['poa_diffuse_total']

        # DIREKTNI del -- g_direct(theta) se spreminja vsako uro (odvisno od trenutnega kota sonca)
        g_direct_hourly = aoi.apply(lambda theta: g_direct(theta, state) if theta < 90 else 0)

        # DIFUZNI del -- g_dif[state] je FIKSNA vrednost, ista skozi celo leto
        q_sol = g_direct_hourly * poa_direct + g_dif[state] * poa_diffuse
        q_sol_by_facade[facade] = q_sol

    q_sol_results[state] = q_sol_by_facade

print("Specificni toplotni dobitek izracunan za vsa 4 stanja, vse fasade, vseh 8760 ur.")

Specificni toplotni dobitek izracunan za vsa 4 stanja, vse fasade, vseh 8760 ur.


In [12]:
annual_solar_kwh_by_state = {}

for state in [0, 25, 50, 100]:
    total_kwh = 0
    for facade in facade_azimuths:
        area = glazing_area_by_orientation[facade]
        q_sol_series = q_sol_results[state][facade]
        total_kwh += (q_sol_series * area).sum() / 1000
    annual_solar_kwh_by_state[state] = total_kwh

print("Letni soncni toplotni dobitek (kWh):")
for state, val in annual_solar_kwh_by_state.items():
    print(f"  {state}%: {val:,.0f} kWh")

Letni soncni toplotni dobitek (kWh):
  0%: 3,350,213 kWh
  25%: 3,248,518 kWh
  50%: 3,079,626 kWh
  100%: 2,591,845 kWh


In [13]:
cost_standard_glass_eur_m2 = 300
cost_spd_glass_eur_m2 = 500
fixed_installation_cost_eur = 10000
avoided_blinds_cost_eur_m2 = 140
avoided_blinds_total_eur = avoided_blinds_cost_eur_m2 * glazing_area

extra_cost_per_m2 = cost_spd_glass_eur_m2 - cost_standard_glass_eur_m2
gross_extra_investment_eur = extra_cost_per_m2 * glazing_area + fixed_installation_cost_eur
total_extra_investment_eur = gross_extra_investment_eur - avoided_blinds_total_eur

activation_mix = {0: 0.40, 25: 0.05, 50: 0.15, 100: 0.40}
weighted_annual_solar_kwh = sum(annual_solar_kwh_by_state[s] * f for s, f in activation_mix.items())
baseline_annual_solar_kwh = annual_solar_kwh_by_state[0]

cooling_hours_fraction = 0.35
Q_constant_annual_kwh = Q_constant/1000 * 8760 * cooling_hours_fraction

annual_kwh_baseline = baseline_annual_solar_kwh + Q_constant_annual_kwh
annual_kwh_weighted = weighted_annual_solar_kwh + Q_constant_annual_kwh

annual_cost_baseline_eur = (annual_kwh_baseline/cooling_cop) * electricity_price_eur_kwh
annual_cost_weighted_eur = (annual_kwh_weighted/cooling_cop) * electricity_price_eur_kwh
annual_cooling_savings_eur = annual_cost_baseline_eur - annual_cost_weighted_eur

print(f"Letni prihranek pri hlajenju: {annual_cooling_savings_eur:,.0f} EUR")
print(f"Neto investicija: {total_extra_investment_eur:,.0f} EUR")

if total_extra_investment_eur > 0 and annual_cooling_savings_eur > 0:
    payback = total_extra_investment_eur / annual_cooling_savings_eur
    print(f"Doba amortizacije: {payback:.1f} let")

Letni prihranek pri hlajenju: 47,467 EUR
Neto investicija: 447,400 EUR
Doba amortizacije: 9.4 let
